# 02 - Preprocessing & Feature Engineering

Questa fase trasforma i dati grezzi in feature pronte per il Machine Learning.

**Step:**
1. Caricamento dati puliti da 01_Data_Ingestion_EDA
2. Data Quality Check (duplicati, missing values)
3. Feature Extraction (Coupon, DTM, YTM, Stress interbancario (Euribor 3m - Tasso BCE), Pendenza a breve (Euribor 1Y - 3M), Differenziale BCE-FED)
4. Gestione delle frequenze miste (Forward-Fill, Shift per Lookahead Bias)
5. Creazione Lagged Features per Time-Series
6. Merge e allineamento temporale finale

## 2.1 - Import & Load Cleaned Data

In [82]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Carica i dataset puliti dal notebook precedente
df_anagrafica_clean = pd.read_csv('./data/df_anagrafica_clean.csv')
df_storico_clean = pd.read_csv('./data/df_storico_clean.csv')
df_macro = pd.read_csv('./data/df_macro.csv', index_col=0, parse_dates=True)

# Assicurati che le date siano datetime
df_storico_clean['referencedate'] = pd.to_datetime(df_storico_clean['referencedate'])
df_anagrafica_clean['redemptiondate'] = pd.to_datetime(df_anagrafica_clean['redemptiondate'], format='%d/%m/%Y', errors='coerce')

print(f"✓ Dataset caricati")
print(f"  Anagrafica: {df_anagrafica_clean.shape}")
print(f"  Storico: {df_storico_clean.shape}")
print(f"  Macro: {df_macro.shape}")

✓ Dataset caricati
  Anagrafica: (305, 10)
  Storico: (189510, 9)
  Macro: (9625, 11)


## 2.2 - Data Quality Check

In [83]:
# 1. Duplicati su ISIN+Data
if 'isincode' in df_storico_clean.columns and 'referencedate' in df_storico_clean.columns:
    n_dupes = df_storico_clean.duplicated(subset=['isincode', 'referencedate']).sum()
    print(f"Duplicati su ISIN+Data nello storico: {n_dupes}")
    
# 2. Missing nelle colonne chiave
print("\nMissing values nelle colonne chiave:")
print(df_storico_clean[['isincode', 'referencedate', 'pricevalue']].isnull().sum())
print("\nAnag missing:")
print(df_anagrafica_clean[['isincode', 'redemptiondate', 'description']].isnull().sum())

Duplicati su ISIN+Data nello storico: 0

Missing values nelle colonne chiave:
isincode         0
referencedate    0
pricevalue       0
dtype: int64

Anag missing:
isincode          0
redemptiondate    0
description       0
dtype: int64


## 2.3 - Feature Extraction: Coupon (Text Mining)

In [84]:
def extract_coupon(description):
    """
    Estrae il tasso di cedola (coupon) dalla descrizione del titolo.
    
    - Zero Coupon: Ritorna 0.0 se contiene 'bot', 'zc', 'zero'
    - Pattern numerico: Cerca "5%" o "3,5%" nella descrizione
    - Pattern EUR: Cerca "EUR 5" nella descrizione
    
    Returns: float (%) oppure np.nan
    """
    if pd.isnull(description):
        return np.nan
    desc = str(description)

    # ZERO COUPON
    if 'bot' in desc.lower() or 'zc' in desc.lower() or 'zero' in desc.lower() or 'ctz' in desc.lower():
        return 0.0

    # Pattern: numeri + %
    match = re.search(r'(\d+[\.,]\d+|\d+)[ ]*%', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
    
    # Pattern: EUR + numeri
    match = re.search(r'[Ee][Uu][Rr][ ]*(\d+[\.,]\d+|\d+)', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
   
    return np.nan

df_anagrafica_clean['coupon'] = df_anagrafica_clean['description'].apply(extract_coupon)

print("--- COUPON EXTRACTION RESULTS ---")
print(f"Coupon estratti: {df_anagrafica_clean['coupon'].notna().sum()}")
print(f"Coupon mancanti: {df_anagrafica_clean['coupon'].isna().sum()}")
print(f"\nCoupon Statistics:")
print(df_anagrafica_clean['coupon'].describe())

# Visualizza alcuni esempi
print("\nEsempi di estrazione:")
display(df_anagrafica_clean[['description', 'coupon']].head(10))

--- COUPON EXTRACTION RESULTS ---
Coupon estratti: 305
Coupon mancanti: 0

Coupon Statistics:
count    305.000000
mean       2.327951
std        1.663578
min        0.000000
25%        0.850000
50%        2.500000
75%        3.450000
max        7.250000
Name: coupon, dtype: float64

Esempi di estrazione:


,description,coupon
0,Btp Fx 3.15% Jun31 Eur,3.15
1,Schatz Fx 2.5% Jun28 Eur,2.50
2,Btp Fx 3.8% Jul36 Eur,3.80
3,Btp Fx 3.3% Jun33 Eur,3.30
4,Bot Zc Apr27 A Eur,0.00
5,Bot Zc Jul26 Q Eur,0.00
6,Bot Zc Sep26 S Eur,0.00
7,Bot Zc Mar27 A Eur,0.00
8,Obligaciones Fx 3.95% Oct56 Eur,3.95
9,Bonos Fx 2.6% May31 Eur,2.60


In [85]:
# Scarta i bond con coupon non estraibile (probabilmente "esotici" sfuggiti al filtro precedente)
n_before = df_anagrafica_clean.shape[0]
df_anagrafica_clean = df_anagrafica_clean.dropna(subset=['coupon']).copy()
print(f"Obbligazioni rimosse per coupon mancante: {n_before - df_anagrafica_clean.shape[0]}")
print(f"Obbligazioni rimanenti: {df_anagrafica_clean.shape[0]}")

Obbligazioni rimosse per coupon mancante: 0
Obbligazioni rimanenti: 305


## 2.4 - Feature Extraction: Days to Maturity (DTM), Yield to Maturity (YTM)

In [86]:
# Merge: Uniamo le feature fisse (cedola, scadenza) al file dei prezzi giornalieri
df_ml = pd.merge(df_storico_clean, 
                 df_anagrafica_clean[['isincode', 'description', 'redemptiondate', 'coupon']], 
                 on='isincode', 
                 how='inner')

print(f"✓ Merge completato: {df_ml.shape[0]} righe")

# DTM = Giorni fino alla scadenza
df_ml['days_to_maturity'] = (df_ml['redemptiondate'] - df_ml['referencedate']).dt.days
df_ml['years_to_maturity'] = df_ml['days_to_maturity'] / 365.25

# Scarta bond già scaduti (DTM <= 0)
n_before = df_ml.shape[0]
df_ml = df_ml[df_ml['days_to_maturity'] > 0].copy()
print(f"Bond scaduti rimossi: {n_before - df_ml.shape[0]}")

print(f"\n--- DTM STATISTICS ---")
print(df_ml[['days_to_maturity', 'years_to_maturity']].describe())

# Preview
print(f"\nSample:")
display(df_ml[['isincode', 'referencedate', 'pricevalue', 'coupon', 'years_to_maturity']].head(10))

✓ Merge completato: 189510 righe
Bond scaduti rimossi: 0

--- DTM STATISTICS ---
       days_to_maturity  years_to_maturity
count     189510.000000      189510.000000
mean        3951.752789          10.819309
std         3649.812827           9.992643
min            7.000000           0.019165
25%         1396.000000           3.822040
50%         2596.000000           7.107461
75%         5434.000000          14.877481
max        18041.000000          49.393566

Sample:


,isincode,referencedate,pricevalue,coupon,years_to_maturity
0,DE0001030708,2023-01-02,83.82,0.0,7.616701
1,DE0001030708,2023-01-03,84.07,0.0,7.613963
2,DE0001030708,2023-01-04,84.62,0.0,7.611225
3,DE0001030708,2023-01-05,84.37,0.0,7.608487
4,DE0001030708,2023-01-06,85.00,0.0,7.605749
5,DE0001030708,2023-01-09,84.98,0.0,7.597536
6,DE0001030708,2023-01-10,84.59,0.0,7.594798
7,DE0001030708,2023-01-11,85.30,0.0,7.592060
8,DE0001030708,2023-01-12,85.67,0.0,7.589322
9,DE0001030708,2023-01-13,85.78,0.0,7.586585


In [87]:
# Yield to Maturity (YTM) ibrido: bisogna distinguere tra zero-coupon e coupon-bearing
# Per i zero-coupon, YTM = (Face Value / Price)^(1/Years to Maturity) - 1
# Per i coupon-bearing, approssimiamo con YTM ≈ (Coupon + (Face Value - Price) / Years to Maturity) / ((Face Value + Price) / 2) 
import numpy as np

def calculate_ytm(row):
    p = row['pricevalue']
    c = row['coupon'] / 100  # Converti da % a decimale
    t = row['years_to_maturity']
    f = 100  # Quasi sempre il valore nominale è 100 per i bond retail

    # Scartiamo i casi impossibili o vicini a scadenza
    if t < 0.8 or p <= 0:
        return np.nan
    
    # Zero-coupon
    if c == 0:
        ytm = ((f / p) ** (1 / t)) - 1
        return ytm * 100  # Converti in percentuale
    
    # Coupon-bearing (approssimazione)
    else:
        # Numeratore: cedola annuale + Capital Gain/Loss annualizzato
        numerator = c + ((f-p) / t)
        # Denominatore: media del valore nominale e del prezzo
        denominator = (f + p) / 2
        ytm_approximated = numerator / denominator
        return ytm_approximated * 100  # Converti in percentuale

df_ml['Yield_to_Maturity'] = df_ml.apply(calculate_ytm, axis=1)

# # Filtra YTM negativi anomali o troppo alti (< -10% o > 50%)
# n_before = df_ml.shape[0]
# df_ml = df_ml[(df_ml['Yield_to_Maturity'] > -10) & (df_ml['Yield_to_Maturity'] < 50)].copy()
# print(f"Scartate {n_before - df_ml.shape[0]} righe con rendimenti matematicamente anomali.")

print("--- Yield to Maturity Statistics ---")
print(df_ml['Yield_to_Maturity'].describe())
print(f"\nYTM negativo: {(df_ml['Yield_to_Maturity'] < 0).sum()}")
print(f"YTM positivo: {(df_ml['Yield_to_Maturity'] >= 0).sum()}")

--- Yield to Maturity Statistics ---
count    185718.000000
mean          0.749999
std           1.585205
min          -5.091635
25%          -0.216024
50%           0.741379
75%           2.008207
max           6.606896
Name: Yield_to_Maturity, dtype: float64

YTM negativo: 59490
YTM positivo: 126228


## 2.5 - Feature Extraction: Stress interbancario (Euribor 3m - Tasso BCE), Pendenza a breve (Euribor 1Y - 3M), Differenziale BCE-FED 

In [88]:
# 1. Stress Interbancario (Euribor 3M - Tasso BCE)
df_macro['interbank_stress_spread'] = df_macro['Euribor_3M'] - df_macro['ECB_Deposit_Rate']

# 2. Pendenza Curva a Breve (Euribor 1Y - Euribor 3M)
df_macro['short_yield_curve_slope'] = df_macro['Euribor_1Y'] - df_macro['Euribor_3M']

# 3. Differenziale BCE-FED (Impatto sul cambio e sui flussi di capitale)
df_macro['bce_fed_spread'] = df_macro['ECB_Deposit_Rate'] - df_macro['FED_Rate']

## 2.6 Separazione dataset: Macro, Macro+Bond

I dati Macro vanno dal 2000 al 2026, tranne XEON SEGA E STOXX50 che vanno dal 2007/08. Quindi li droppiamo così da avere più dati possibili.
Mentre nel dataset Bond+Macro dove i bond vanno dal 2023 al 2026 possiamo considerarli.

In [89]:
# MACRO ONLY
df_macro_long = df_macro.drop(columns=['XEON','SEGA','STOXX50'])

# Salviamo il dataset per i notebook successivi 
df_macro_long.to_csv('./data/df_macro_long.csv')

In [90]:
# MACRO + BONDS
# Prepara il DataFrame macro per il merge
df_macro_reset = df_macro.reset_index()
df_macro_reset.columns = df_macro_reset.columns.str.lower()
# Crea la colonna 'referencedate' a partire dall'indice reset (qui chiamato 'index') e rimuovi l'originale
df_macro_reset['referencedate'] = pd.to_datetime(df_macro_reset['index'])
df_macro_reset = df_macro_reset.drop(columns=['index'])

# Assicurati che le date siano datetime
df_ml['referencedate'] = pd.to_datetime(df_ml['referencedate'])

# Merge: left join per mantenere tutte le righe di df_ml e aggiungere i dati macro corrispondenti
df_bond_macro = pd.merge(df_ml, df_macro_reset, 
                        left_on='referencedate', right_on='referencedate', 
                        how='left')


# Salviamo il dataset per i notebook successivi
df_bond_macro.to_csv('./data/df_bond_macro.csv', index=False)


## 2.7 - Lagged Features per dataset Macro only

In [91]:
if df_macro_long is not None:
    lags = [7, 21, 63, 126, 252] # 1w, 1m, 3m, 6m, 1y in giorni di borsa
    for col in df_macro_long.columns:
        for lag in lags:
            df_macro_long[f'{col}_lag_{lag}d'] = df_macro_long[col].shift(lag)

df_macro_long.dropna(inplace=True)
df_macro_long.to_csv('./data/df_macro_long_lagged.csv')
display(df_macro_long.head())

,ECB_Deposit_Rate,ECB_MRO_Rate,FED_Rate,VIX,ESI_Index,Euribor_3M,Euribor_1Y,HICP_Euroarea,interbank_stress_spread,short_yield_curve_slope,...,short_yield_curve_slope_lag_7d,short_yield_curve_slope_lag_21d,short_yield_curve_slope_lag_63d,short_yield_curve_slope_lag_126d,short_yield_curve_slope_lag_252d,bce_fed_spread_lag_7d,bce_fed_spread_lag_21d,bce_fed_spread_lag_63d,bce_fed_spread_lag_126d,bce_fed_spread_lag_252d
2000-09-09,3.5,4.25,6.50,18.46,-3.7,4.85281,5.467185,59.29,1.35281,0.614375,...,0.614375,0.621091,0.870831,1.153452,2.355396,-3.02,-3.19,-3.17,-4.28,-5.03
2000-09-10,3.5,4.25,6.50,18.46,-3.7,4.85281,5.467185,59.29,1.35281,0.614375,...,0.614375,0.621091,0.870831,1.153452,2.355396,-3.02,-3.19,-3.17,-4.28,-5.03
2000-09-11,3.5,4.25,6.50,18.40,-3.7,4.85281,5.467185,59.29,1.35281,0.614375,...,0.614375,0.621091,0.870831,1.153452,2.355396,-3.02,-3.23,-3.26,-4.28,-5.03
2000-09-12,3.5,4.25,6.47,18.59,-3.7,4.85281,5.467185,59.29,1.35281,0.614375,...,0.614375,0.621091,0.870831,1.153452,2.355396,-3.11,-3.17,-3.14,-4.28,-5.03
2000-09-13,3.5,4.25,6.47,18.32,-3.7,4.85281,5.467185,59.29,1.35281,0.614375,...,0.614375,0.621091,0.870831,1.153452,2.355396,-3.06,-3.25,-3.13,-4.28,-5.03


## 2.8 - Lagged Features per dataset Macro + Bonds

In [92]:
display(df_bond_macro.columns)

Index(['isincode', 'marketcode', 'referencedate', 'endvaluedate', 'pricetype',
       'pricevalue', 'volume', 'mintoday', 'maxtoday', 'description',
       'redemptiondate', 'coupon', 'days_to_maturity', 'years_to_maturity',
       'Yield_to_Maturity', 'ecb_deposit_rate', 'ecb_mro_rate', 'fed_rate',
       'vix', 'esi_index', 'euribor_3m', 'euribor_1y', 'hicp_euroarea',
       'stoxx50', 'xeon', 'sega', 'interbank_stress_spread',
       'short_yield_curve_slope', 'bce_fed_spread'],
      dtype='object')

In [93]:
# Applica per ogni ISIN
df_bond_macro = df_bond_macro.sort_values(['isincode', 'referencedate']).reset_index(drop=True)

# Per ogni ISIN, crea lagged features per catturare la memoria storica
    # Ad es: prezzo di ieri, 3 giorni fa, 7 giorni fa, ecc.
lags = [1, 3, 7, 15, 30, 60]

for lag in lags:
    # Usando groupby().shift() pandas non usa cicli lenti, lo fa a livello C/Cython!
    df_bond_macro[f'pricevalue_lag_{lag}d'] = df_bond_macro.groupby('isincode')['pricevalue'].shift(lag)

# Creiamo lag di 7,15,30 giorni anche per le macro variabili fondamentali in modo da catturare l'effetto ritardato delle condizioni macroeconomiche sui prezzi dei bond
macro_to_lag = ['volume', 'Yield_to_Maturity', 'ecb_deposit_rate', 'ecb_mro_rate', 'fed_rate',
       'vix', 'esi_index', 'euribor_3m', 'euribor_1y', 'hicp_euroarea',
       'stoxx50', 'xeon', 'sega', 'interbank_stress_spread',
       'short_yield_curve_slope', 'bce_fed_spread',]
for col in macro_to_lag:
    if col in df_bond_macro.columns:
        df_bond_macro[f'{col}_lag_7d'] = df_bond_macro.groupby('isincode')[col].shift(7)
        df_bond_macro[f'{col}_lag_15d'] = df_bond_macro.groupby('isincode')[col].shift(15)
        df_bond_macro[f'{col}_lag_30d'] = df_bond_macro.groupby('isincode')[col].shift(30)

# Scarta le righe con NaN nei lag (i primi 60 giorni per ogni ISIN)
lag_cols = [f'pricevalue_lag_{lag}d' for lag in lags]
df_bond_macro = df_bond_macro.dropna(subset=lag_cols).copy()

print(f"✓ Lagged features creati")
print(f"Dataset dopo lagged features: {df_bond_macro.shape}")
print(f"\nColonne lag:")
lag_cols = [col for col in df_bond_macro.columns if 'lag' in col]
print(lag_cols)

df_bond_macro.to_csv('./data/df_bond_macro_lagged.csv', index=False)

✓ Lagged features creati
Dataset dopo lagged features: (171683, 83)

Colonne lag:
['pricevalue_lag_1d', 'pricevalue_lag_3d', 'pricevalue_lag_7d', 'pricevalue_lag_15d', 'pricevalue_lag_30d', 'pricevalue_lag_60d', 'volume_lag_7d', 'volume_lag_15d', 'volume_lag_30d', 'Yield_to_Maturity_lag_7d', 'Yield_to_Maturity_lag_15d', 'Yield_to_Maturity_lag_30d', 'ecb_deposit_rate_lag_7d', 'ecb_deposit_rate_lag_15d', 'ecb_deposit_rate_lag_30d', 'ecb_mro_rate_lag_7d', 'ecb_mro_rate_lag_15d', 'ecb_mro_rate_lag_30d', 'fed_rate_lag_7d', 'fed_rate_lag_15d', 'fed_rate_lag_30d', 'vix_lag_7d', 'vix_lag_15d', 'vix_lag_30d', 'esi_index_lag_7d', 'esi_index_lag_15d', 'esi_index_lag_30d', 'euribor_3m_lag_7d', 'euribor_3m_lag_15d', 'euribor_3m_lag_30d', 'euribor_1y_lag_7d', 'euribor_1y_lag_15d', 'euribor_1y_lag_30d', 'hicp_euroarea_lag_7d', 'hicp_euroarea_lag_15d', 'hicp_euroarea_lag_30d', 'stoxx50_lag_7d', 'stoxx50_lag_15d', 'stoxx50_lag_30d', 'xeon_lag_7d', 'xeon_lag_15d', 'xeon_lag_30d', 'sega_lag_7d', 'sega_la

## 2.9 - Analisi Preprocessed Data for Modeling

In [94]:
# MACRO + BONDS LAGGED
print(f"\nRiepilogo Dataset Macro-Bond:")
print(f"  Numero di righe: {df_bond_macro.shape[0]}")
print(f"  Numero di colonne: {df_bond_macro.shape[1]}")
print(f"  Numero di ISIN unici: {df_bond_macro['isincode'].nunique()}")
print(f"  Numero di NaN dopo lag: {df_bond_macro.isnull().sum().sum()}")
print(f"  Data range: {df_bond_macro['referencedate'].min()} - {df_bond_macro['referencedate'].max()}")
print(f"\nFeatures:")
for col in df_bond_macro.columns:
    if col != 'isincode' and col != 'referencedate':
        print(f"  - {col}")


Riepilogo Dataset Macro-Bond:
  Numero di righe: 171683
  Numero di colonne: 83
  Numero di ISIN unici: 290
  Numero di NaN dopo lag: 167472
  Data range: 2023-03-29 00:00:00 - 2026-05-07 00:00:00

Features:
  - marketcode
  - endvaluedate
  - pricetype
  - pricevalue
  - volume
  - mintoday
  - maxtoday
  - description
  - redemptiondate
  - coupon
  - days_to_maturity
  - years_to_maturity
  - Yield_to_Maturity
  - ecb_deposit_rate
  - ecb_mro_rate
  - fed_rate
  - vix
  - esi_index
  - euribor_3m
  - euribor_1y
  - hicp_euroarea
  - stoxx50
  - xeon
  - sega
  - interbank_stress_spread
  - short_yield_curve_slope
  - bce_fed_spread
  - pricevalue_lag_1d
  - pricevalue_lag_3d
  - pricevalue_lag_7d
  - pricevalue_lag_15d
  - pricevalue_lag_30d
  - pricevalue_lag_60d
  - volume_lag_7d
  - volume_lag_15d
  - volume_lag_30d
  - Yield_to_Maturity_lag_7d
  - Yield_to_Maturity_lag_15d
  - Yield_to_Maturity_lag_30d
  - ecb_deposit_rate_lag_7d
  - ecb_deposit_rate_lag_15d
  - ecb_deposit_rat

In [95]:
# MACRO ONLY LAGGED
print(f"\nRiepilogo Dataset Macro-Only:")
print(f"  Numero di righe: {df_macro_long.shape[0]}")
print(f"  Numero di colonne: {df_macro_long.shape[1]}")
print(f"  Numero di NaN dopo lag: {df_macro_long.isnull().sum().sum()}")
print(f"  Data range: {df_macro_long.index.min()} - {df_macro_long.index.max()}")
print(f"\nFeatures:")
for col in df_macro_long.columns:
    if col != 'isincode' and col != 'referencedate':
        print(f"  - {col}")


Riepilogo Dataset Macro-Only:
  Numero di righe: 9373
  Numero di colonne: 66
  Numero di NaN dopo lag: 0
  Data range: 2000-09-09 00:00:00 - 2026-05-08 00:00:00

Features:
  - ECB_Deposit_Rate
  - ECB_MRO_Rate
  - FED_Rate
  - VIX
  - ESI_Index
  - Euribor_3M
  - Euribor_1Y
  - HICP_Euroarea
  - interbank_stress_spread
  - short_yield_curve_slope
  - bce_fed_spread
  - ECB_Deposit_Rate_lag_7d
  - ECB_Deposit_Rate_lag_21d
  - ECB_Deposit_Rate_lag_63d
  - ECB_Deposit_Rate_lag_126d
  - ECB_Deposit_Rate_lag_252d
  - ECB_MRO_Rate_lag_7d
  - ECB_MRO_Rate_lag_21d
  - ECB_MRO_Rate_lag_63d
  - ECB_MRO_Rate_lag_126d
  - ECB_MRO_Rate_lag_252d
  - FED_Rate_lag_7d
  - FED_Rate_lag_21d
  - FED_Rate_lag_63d
  - FED_Rate_lag_126d
  - FED_Rate_lag_252d
  - VIX_lag_7d
  - VIX_lag_21d
  - VIX_lag_63d
  - VIX_lag_126d
  - VIX_lag_252d
  - ESI_Index_lag_7d
  - ESI_Index_lag_21d
  - ESI_Index_lag_63d
  - ESI_Index_lag_126d
  - ESI_Index_lag_252d
  - Euribor_3M_lag_7d
  - Euribor_3M_lag_21d
  - Euribor_3M_